# Regularization: linear against Ridge against Lasso

The agency liked the 2.2 model but asked why one coefficient is so large: a poor-quality garage (`GarageQual` = `Po`) cuts the predicted price by 39.2%, and only one training house has that value. You fit the same pipeline three ways, plot a learning curve for each, and recommend one model.

## The starter

The cell repeats the 2.2 setup: D01 6.4's decisions, the 75 / 25 split, `pre` (and `make_pre(X)` to build a fresh copy), a scorer `rmse_dollars` that reports errors in dollars, and the folds `cv = KFold(5, shuffle=True, random_state=42)`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, root_mean_squared_error
from sklearn.model_selection import KFold, learning_curve, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/ames.csv", keep_default_na=False, na_values=[""])
df = df[df["GrLivArea"] <= 4000]
y = np.log(df["SalePrice"])
X = df.drop(columns=["Id", "SalePrice", "SaleType", "SaleCondition"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

STRUCTURAL = ["Alley", "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
              "FireplaceQu", "GarageType", "GarageFinish", "GarageQual", "GarageCond", "PoolQC",
              "Fence", "MiscFeature"]


def make_pre(X):
    """D01 6.4's three-block ColumnTransformer for the columns X has."""
    num_cols = list(X.select_dtypes(include="number").columns)
    cat_cols = list(X.select_dtypes(include=["str", "object"]).columns)
    structural = [c for c in STRUCTURAL if c in cat_cols]
    other_cat = [c for c in cat_cols if c not in STRUCTURAL]
    blocks = [
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
        ("none", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"),
                               OneHotEncoder(handle_unknown="ignore", sparse_output=False)), structural),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                              OneHotEncoder(handle_unknown="ignore", sparse_output=False)), other_cat),
    ]
    return ColumnTransformer([b for b in blocks if b[2]])


pre = make_pre(X)


def dollar_rmse(y_log_true, y_log_pred):
    """RMSE in dollars. The clip stops one wild prediction on a small training set from overflowing exp."""
    return root_mean_squared_error(np.exp(y_log_true), np.exp(np.clip(y_log_pred, 9, 15)))


rmse_dollars = make_scorer(dollar_rmse, greater_is_better=False)

# On the smallest learning-curve sizes no training house has a pool, so the imputer warns that
# PoolQC has no observed values and skips it. The warning is expected; this line hides it.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")
cv = KFold(5, shuffle=True, random_state=42)
print(len(X_train), "training houses,", len(X_test), "test houses")

## Step 1 · The 2.2 model again

Refit `make_pipeline(pre, LinearRegression())` and print its test RMSE in dollars and the coefficient of `none__GarageQual_Po`. Find it with `pd.Series(model[-1].coef_, index=model[0].get_feature_names_out())`.

## Step 2 · Ridge

Replace `LinearRegression()` with `RidgeCV(alphas=np.logspace(-3, 3, 13), cv=cv)`. Print the chosen `alpha_`, the test RMSE and the same coefficient.

## Step 3 · Lasso

Replace it with `LassoCV(alphas=np.logspace(-5, -1, 13), cv=cv, max_iter=50000, random_state=42)`. Print `alpha_`, the test RMSE, the same coefficient, and how many coefficients are not zero. Count a coefficient as zero when its absolute value is below 1e-10: `(coef != 0).sum()` also counts two values near 1e-16, which are rounding leftovers.

## Step 4 · "Is the plain model memorizing our houses?"

Translate: `learning_curve` for each of the three models, on the training rows only, with `cv=cv`, `train_sizes=np.linspace(0.15, 1.0, 8)`, `scoring=rmse_dollars`, `shuffle=True, random_state=42`. Plot training and validation RMSE against the number of rows, one chart per model. Use `Ridge(alpha=...)` and `Lasso(alpha=...)` with the alphas from steps 2 and 3.

## Step 5 · The gap at the largest size

For each model, the gap between validation and training RMSE at the largest size.

## Step 6 · Recommend one model

Two sentences for the agency: which one, the test RMSE, and one thing it does that the others do not.

## Bonus · ElasticNet

## Check yourself